In [ ]:
import pandas as pd
import numpy as np
from copy import deepcopy
import ast
import polars as pl

In [ ]:
import matplotlib.pyplot as plt
import fastplot
from matplotlib.ticker import MaxNLocator
import matplotlib.dates as mdates
import matplotlib.lines as mlines
from matplotlib.dates import DateFormatter
import seaborn as sns
%matplotlib inline

In [ ]:
from novatrak.monic import Monic_with_noise
from novatrak.track import ClusterTracker

In [ ]:
from datetime import datetime, timedelta
START = '20250401'
def get_day(i,start = START):
    start = datetime.strptime(start, '%Y%m%d')
    day = start+timedelta(days=i)
    day = day.strftime('%Y%m%d')
    return day

In [ ]:
def ecdf(arr):
    val, nums = np.unique(arr, return_counts=True)
    ecdf = np.cumsum(nums) / np.sum(nums)
    return val, ecdf

In [ ]:
darknet = '<DARKNET_NAME>'

## Cluster transition (2 batches)

In [ ]:
day_0 = pd.read_csv(f'results_clusters/{darknet}/{darknet}_{START}.csv',index_col = 'ip')
emerged = []
current_day = START
for i in range(1,30):
    next_day = get_day(i)
    day_1 = pd.read_csv(f'results_clusters/{darknet}/{darknet}_{next_day}.csv',index_col = 'ip')

    #run MONIC
    monic = Monic_with_noise()
    monic.fit(day_0,day_1)
    

    #save transition and other info to csv
    df_trans = monic.get_transition_df()
    df_cluster_info = day_0.groupby("cluster").agg(
        avg_sh=("sh", "mean"),
        ip=("sh", lambda x: list(x.index)),
        num_ip = ("sh","size")
        )
    df_cluster_info = df_cluster_info[df_cluster_info.index > -1]
    df_trans = df_trans.merge(df_cluster_info, on='cluster')
    df_trans['emerge'] = df_trans.cluster.apply(lambda x: True if x in emerged else False)
    PATH_OUTPUT = f'results_clusters/{darknet}/transition/{darknet}_transition_{current_day}.csv'
    df_trans = df_trans.sort_values('cluster')
    # df_trans.to_csv(PATH_OUTPUT,index=False)
    
    # update day_0 data
    emerged = deepcopy(list(monic.emerged))
    day_0 = deepcopy(day_1)
    current_day = next_day

## Long-term tracking

In [ ]:
tracker = ClusterTracker()
for i in range(30):
    DAY = get_day(i)
    df = pd.read_csv(f'results_clusters/ts/transition/ts_transition_{DAY}.csv')
    df["matching"] = df["matching"].apply(ast.literal_eval)
    df["ip"] = df["ip"].apply(ast.literal_eval)
    df = df.sort_values('cluster')
    tracker.track_single_day(DAY,df)
    
tracker.update_df_matching()

tracked_df = pd.DataFrame()
for i in range(30):
    DAY = get_day(i)
    df_day = tracker.all_dfs[DAY]
    df_day['day'] = DAY
    tracked_df = pd.concat([tracked_df,df_day])

## Results and plots

In [ ]:
day_ext_transition = tracked_df.groupby(['transition','day']).size().unstack(fill_value=0).to_dict(orient="index")
day_ext_transition_list = {}
for trans in day_ext_transition.keys():
    num_list = []
    for i in range(30):
        DAY = get_day(i)
        num_list.append(day_ext_transition[trans][DAY])
    day_ext_transition_list[trans] = num_list
day_ext_transition_list['Survived'] = day_ext_transition_list.pop('survived')
day_ext_transition_list['Inactive'] = day_ext_transition_list.pop('inactive')
day_ext_transition_list['Disappeared'] = day_ext_transition_list.pop('disappeared')
day_ext_transition_list['Absorbed'] = day_ext_transition_list.pop('absorbed')
day_ext_transition_list['Split'] = day_ext_transition_list.pop('splitted')

In [ ]:
ticks_dates = [datetime(2025,4,1)+timedelta(days = i) for i in range(30)]

In [ ]:
num_clusters = tracked_df.groupby("day").size()

In [ ]:
def plot_bar_transition(plt):
    width = 0.74
    bottom = np.zeros(29)
        
    for boolean, weight_count in day_ext_transition_list.items():
        p = plt.bar(ticks_dates[:-1], weight_count[:-1], width, label=boolean, bottom=bottom)
        bottom += weight_count[:-1]
    # plt.bar(ticks_dates[-1],0,bottom = 0)    
    plt.legend(borderpad=0.1,handletextpad = 0.2,loc = 4,ncols = 2, columnspacing=0.1)
    # plt.grid(linestyle = 'dotted')
    plt.xlabel('Date')
    plt.ylabel('Number of clusters')
    dates = [datetime(2025, 4, 1) + timedelta(days=i) for i in range(29) if i%2==1]
    locator = mdates.AutoDateLocator(minticks=9, maxticks=10)
    formatter = mdates.ConciseDateFormatter(locator)
    ax=plt.gca()
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    ax.set_xticks(dates)

plot = fastplot.plot(None,  None, mode = 'callback', 
                     callback = lambda plt: plot_bar_transition(plt),
                     figsize=(5, 3.5), fontsize=14, style='latex')
# plot.savefig('results_plots/bar_transition_ext.pdf')
plot.show()

In [ ]:
num_reocur = []
num_novelty = []
df_day = tracker.all_dfs['20250401']
day_trans = df_day.value_counts('transition').to_dict()
max_gid = df_day.gid.values.max()
for i in range(1, 30):
    DAY = get_day(i)
    df_day = tracker.all_dfs[DAY]
    day_trans = df_day.value_counts('transition').to_dict()
    df_emg = df_day[df_day.emerge]
    # print(df_emg.shape[0])
    df_nov = df_emg[df_emg.gid>max_gid]
    num_novelty.append(df_nov.shape[0])
    num_reocur.append(df_emg.shape[0] - df_nov.shape[0])
    max_gid = df_day.gid.values.max()

In [ ]:
def plot_bar_novelty(plt):
    width = 0.74
    plt.bar(ticks_dates[1:], num_novelty, width, bottom=np.zeros(29),color = 'orange',label = 'Novelty')
    plt.bar(ticks_dates[1:], num_reocur, width, bottom=num_novelty,label = 'Reocurred',color = 'orange',alpha = 0.4,)
    # plt.bar(ticks_dates[-1],0,bottom = 0)    
    plt.legend(borderpad=0.1,handletextpad = 0.2,loc = 2,ncols = 2, columnspacing=0.1)
    # plt.grid(linestyle = 'dotted')
    plt.xlabel('Date')
    plt.ylabel('Number of clusters')
    dates = [datetime(2025, 4, 1) + timedelta(days=i) for i in range(1,30) if i%2==0]
    locator = mdates.AutoDateLocator(minticks=9, maxticks=10)
    formatter = mdates.ConciseDateFormatter(locator)
    ax=plt.gca()
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    ax.set_xticks(dates)

plot = fastplot.plot(None,  None, mode = 'callback', 
                     callback = lambda plt: plot_bar_novelty(plt),
                     figsize=(5, 3.5), fontsize=14, style='latex')
# plot.savefig('results_plots/bar_emerged.pdf')
plot.show()

### transition plot

In [ ]:
def plot_transitions(plt):
    markercolor = 'b'
    symbol = '_'
    start_date = '20250401'
    num_days = 30
    show_trans_lines = True
    drange = pd.date_range(start_date, periods=num_days, freq="D")
    # fig, ax = plt.subplots()
    for i in range(num_days-1):
        DATE = get_day(i,start_date)
        print(DATE)
        NEXT_DAY = get_day(1,DATE)
        cluster_day = tracker.all_dfs[DATE]
        cluster_day['date'] = drange[i]
        # gid_day = cluster_day.gid.values

        # survived
        survived = cluster_day[cluster_day.transition == 'survived']
        plt.scatter(survived['date'].values, survived['gid'].values, marker = '_', color='b')

        # inactive
        inactive = cluster_day[cluster_day.transition == 'inactive']
        plt.scatter(inactive['date'].values, inactive['gid'].values, marker = '_', color='red')

        # disappeared
        disappeared = cluster_day[cluster_day.transition == 'disappeared']
        plt.scatter(disappeared['date'].values, disappeared['gid'].values, marker = 'x', color='b')

        # absorbed
        absorbed = cluster_day[cluster_day.transition == 'absorbed']
        plt.scatter(absorbed['date'].values, absorbed['gid'].values, marker = '>', color='b')
        if show_trans_lines:
            for _, row in absorbed.iterrows():
                y1 = row["gid"]
                matched = row["matching_gid"]
                x1 = row["date"]
                x2 = drange[i+1]
                for y2 in matched:
                    plt.plot([x1, x2], [y1, y2], color = 'darkorange',linestyle = 'dotted')

        # split
        splitted = cluster_day[cluster_day.transition == 'splitted']
        plt.scatter(splitted['date'].values, splitted['gid'].values, marker = '<', color='b')
        if show_trans_lines:
            for _, row in splitted.iterrows():
                y1 = row["gid"]
                matched = row["matching_gid"]
                x1 = row["date"]
                x2 = drange[i+1]
                for y2 in matched:
                    # print('split')
                    plt.plot([x1, x2], [y1, y2], color = 'green',linestyle = 'dotted')
    # last day don't know transitions
    last_day_trans = tracker.all_dfs[tracker._get_day(num_days-1,start_date)]
    last_day_trans['date']= drange[-1]
    plt.scatter(last_day_trans['date'],last_day_trans['gid'], marker = '_', color='b')
    plt.ylabel('Activity ID')
    dates = [datetime(2025, 4, 1) + timedelta(days=i) for i in range(30)if i%2==1]
    # dates = [datetime(2025, 4, 1) + timedelta(days=i) for i in range(30) if i%2==1]
    locator = mdates.AutoDateLocator(minticks=9, maxticks=10)
    formatter = mdates.ConciseDateFormatter(locator)
    ax=plt.gca()
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    ax.set_xticks(dates)
    ax.set_xlim([datetime(2025, 3, 31,12),datetime(2025, 4, 30,12)])
    survived_mark = mlines.Line2D([], [], color=markercolor, marker=symbol, linestyle='None', label='Survived')
    inactive_mark = mlines.Line2D([], [], color='r', marker=symbol, linestyle='None', label='Inactive')
    splitted_mark = mlines.Line2D([], [], color=markercolor, marker='<', linestyle='None', label='Split')
    disappeared_mark = mlines.Line2D([], [], color=markercolor, marker='x', linestyle='None', label='Disappeared')
    absorbed_mark = mlines.Line2D([], [], color=markercolor, marker='>', linestyle='None', label='Absorbed')
    
    # red_line = mlines.Line2D([], [], color='r', label = "Clusters of other labels")
    ax.legend(handles=[survived_mark, absorbed_mark, splitted_mark, disappeared_mark, inactive_mark], #survived_mark, disappeared_mark, absorbed_mark, inactive_mark, splitted_mark
              # fontsize='small',
              # bbox_to_anchor=(0, 1.005, 1., .102),
              ncols = 5,borderpad=0.2,handletextpad = 0, columnspacing=0.1,
              loc='upper center'#loc=9
             )

In [ ]:
plot = fastplot.plot(None,  None, mode = 'callback', 
                     callback = lambda plt: plot_transitions(plt),
                     figsize=(9, 5), fontsize=14, style='latex')
plot.savefig('results_plots/tracking_30_days.pdf')
plot.show()